**1. What is Delta Lake?**

Delta Lake is a **storage layer built on top of Parquet** that adds ACID transactions, schema enforcement/evolution, time travel, and operations such as UPDATE, DELETE, and MERGE.

**A simple way to remember:**
````
Delta = Parquet files + transaction log + additional capabilities
````

**2. Delta vs plain Parquet — what's the difference?**

- Parquet --> is a **columnar file format**. By itself, it does not provide transaction management.

- Delta Lake --> stores the underlying data in Parquet but adds a transaction log (_delta_log), which enables:

    -     ACID transactions
    -     UPDATE / DELETE / MERGE
    -     Time Travel
    -     Schema enforcement/evolution
    -     Table version history

For interviews, the key phrase is:

Delta Lake doesn't replace Parquet. It uses Parquet for data storage and adds a transactional layer on top.

**3. What does ACID mean?**

**A — Atomicity:** Transaction either completely succeeds or completely fails.

**C — Consistency:** Data remains in a valid state before and after a transaction.

**I — Isolation:** Concurrent transactions don't interfere with each other.

**D — Durability:** Once a transaction is committed, the change persists.

For ETL, an easy example is: if you're writing millions of records and the job fails midway, Delta prevents readers from seeing a partially committed transaction.

**4. What is the Delta transaction log?**

The Delta transaction log **(_delta_log)** records all changes made to a Delta table and allows Delta Lake to **track the current state and previous versions of the table**.

It contains information about things such as:

        files added
        files removed
        table metadata
        schema changes
        transaction/version information

This is what enables ACID transactions and Time Travel.


**5. What does MERGE do?**

MERGE allows us to conditionally INSERT, UPDATE, or DELETE target records by matching a source dataset against a target table.
````
target.alias("t").merge(
    source.alias("s"),
    "t.custId = s.custId"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
````

**Using your Informatica background:**
````
MERGE ≈ Lookup target → identify existing/new → Router → Update Strategy → INSERT/UPDATE
````


**6. What's the difference between UPDATE and MERGE?**

- **Update** - Modifies existing records that satisfy a condition.
- **Merge**- Compares source and target using a matching condition and can perform different actions depending on whether a record matches.
    ````
        Matched → UPDATE
        Not matched → INSERT
    ````
    - **.whenMatchedUpdateAll()** means for every source/target row that satisfies the MERGE matching condition, copy all source columns into the corresponding target columns.
-     It doesn't check whether individual column values changed.

    ````
    for example- 
       .whenMatchedUpdate(set={
            "status": "s.status",
            "city": "s.city"
        })
    ````
    updates only those specified columns. 

    whereas:
    ````
    .whenMatchedUpdateAll() 
    ````
    sets all corresponding target columns from the source for matched records.


**7. What is schema enforcement?**

Schema enforcement prevents **incompatible data** from being written to a Delta table when the incoming data doesn't match the expected schema. 

**8. What is schema evolution?**

Schema evolution allows the target Delta schema to evolve when the incoming data contains schema changes, such as adding a new column.
````
Old:
custId, name, city

New:
custId, name, city, email
````
Rather than manually recreating the table, schema evolution can allow email to be incorporated.


**9. What is Delta Time Travel?**

Time Travel lets us query a previous version of a Delta table, which is useful for auditing, debugging, comparing data before/after a load, or recovering from an accidental change.

````
select * from cutsomer360.customer_master VERSION AS OF 3
or
select * from cutsomer360.customer_master@v3
````


**10. What does DESCRIBE HISTORY tell you?**
registers all the changes that  happend on the table. 
````
DESCRIBE HISTORY customer360.customer_master
````
It gives information such as:

- Version
- Timestamp
- Operation (WRITE, MERGE, UPDATE, DELETE, etc.)
- User
- Operation parameters
- Metrics

And yes, version 0 is typically the initial creation/write.

**11. Give a real ETL use case where you would use MERGE.**

Suppose I receive a daily customer file. Some customers already exist in customer_master and have changed attributes, while some customers are new. I can MERGE the daily file into the Delta table using custId as the matching key. Existing customers are updated and new customers are inserted.

````
Source                    Target

1002 Mary Inactive  →     1002 Mary Active
1003 Steve McKinney →     1003 Steve Frisco
1004 Sarah Irving   →     doesn't exist

MERGE result:

1002 → UPDATE
1003 → UPDATE
1004 → INSERT
````

This is a very natural example for you because you can explain that you previously implemented the same ETL pattern in Informatica with Lookup → Router → Update Strategy, while Delta provides MERGE directly.


**12. Senior scenario**

You receive 50 million customer changes daily. Some are inserts, some are updates, and most customers haven't changed. How would you design the Delta load without overwriting the entire customer table every day? 


I would implement an incremental Delta MERGE rather than doing a full overwrite. I would first process only the incoming changed records and deduplicate them if multiple records exist for the same customer. Then I would MERGE the incremental dataset into the customer Delta table using the customer ID as the business key.

For matched customers whose relevant attributes changed, I would UPDATE them. For customers that don't exist in the target, I would INSERT them. Unchanged customers would not need to be rewritten logically by the ETL.

I would also consider table layout and optimization based on the size of the target and the access patterns, because at this scale MERGE performance matters.

````
50M daily source
      ↓
clean / deduplicate / validate
      ↓
incremental changes
      ↓
MERGE on custId
      ↓
┌───────────────┬────────────────┐
│ MATCHED       │ NOT MATCHED    │
│ + changed     │                │
│ UPDATE        │ INSERT         │
└───────────────┴────────────────┘
      ↓
customer_master Delta
````